# Scraping and geocoding medical facilities in Belarus

This notebook collects medical-facility data from Healthcare.by, cleans the scraped records, geocodes facility addresses, and assigns administrative regions and districts.

API keys are read from a local `.env` file and are not stored in the notebook.

## 1. Imports

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
import re

## 2. Healthcare.by directory pages

The four directory categories are feldsher-midwife stations, outpatient clinics, hospitals, and polyclinics.

In [3]:
fap = "https://healthcare.by/search.php?typ=707&sort=1&page=" 
amb = "https://healthcare.by/search.php?typ=600&sort=1&page=" 
hospital = "https://healthcare.by/search.php?typ=603&sort=1&page="
polyclinic = "https://healthcare.by/search.php?typ=598&sort=1&page="

In [4]:
links = [fap, amb, hospital, polyclinic]

## 3. Classify facility types

Facility names are classified using keywords and common abbreviations.

In [5]:
def get_facility_type(name):
    name = name.lower().replace("ё", "е")

    facility_types = {
        "ФАП": ("FAP", ("фап", "фельдшерско-акушерский пункт")),
        "Амбулатория": ("Outpatient Clinic", ("амбулатория", "АВОП", "авоп", "ВАП", "вап")),
        "Больница": ("Hospital", ("больница",)),
        "Поликлиника": ("Polyclinic", ("поликлиника",)),
    }

    for facility_type, (type_eng, keywords) in facility_types.items():
        if any(keyword in name for keyword in keywords):
            return facility_type, type_eng

    return None, None

## 4. Scrape facility listings

The code visits every result page in each category and collects the facility name, directory link, settlement, and facility type.

In [6]:
rows = []

for category_url in links:
    base_url = category_url.split("&page=")[0]

    response = requests.get(base_url, params={"page": 1})
    doc = BeautifulSoup(response.text, "html.parser")

    page_numbers = []

    for page_link in doc.select('a[href*="page="]'):
        match = re.search(r"page=(\d+)", page_link.get("href", ""))

        if match:
            page_numbers.append(int(match.group(1)))

    last_page = max(page_numbers, default=1)
    print("Количество страниц:", last_page)

    for page in range(1, last_page + 1):
        response = requests.get(
            base_url,
            params={"page": page},
            timeout=30
        )
        response.raise_for_status()

        print(response.url)

        doc = BeautifulSoup(response.text, "html.parser")
        items = doc.select("li")

        for item in items:
            facility_link = item.select_one("a")

            if facility_link is None:
                continue

            name = facility_link.get_text(" ", strip=True)
            facility_type, type_eng = get_facility_type(name)

            row = {
                "facility_title": name,
                "link": f"https://healthcare.by/{facility_link['href']}",
                "settlement": facility_link.next_sibling.strip(" \xa0()"),
                "facility_type": facility_type,
                "facility_type_eng": type_eng,
            }

            rows.append(row)

Количество страниц: 105
https://healthcare.by/search.php?typ=707&sort=1&page=1
https://healthcare.by/search.php?typ=707&sort=1&page=2
https://healthcare.by/search.php?typ=707&sort=1&page=3
https://healthcare.by/search.php?typ=707&sort=1&page=4
https://healthcare.by/search.php?typ=707&sort=1&page=5
https://healthcare.by/search.php?typ=707&sort=1&page=6
https://healthcare.by/search.php?typ=707&sort=1&page=7
https://healthcare.by/search.php?typ=707&sort=1&page=8
https://healthcare.by/search.php?typ=707&sort=1&page=9
https://healthcare.by/search.php?typ=707&sort=1&page=10
https://healthcare.by/search.php?typ=707&sort=1&page=11
https://healthcare.by/search.php?typ=707&sort=1&page=12
https://healthcare.by/search.php?typ=707&sort=1&page=13
https://healthcare.by/search.php?typ=707&sort=1&page=14
https://healthcare.by/search.php?typ=707&sort=1&page=15
https://healthcare.by/search.php?typ=707&sort=1&page=16
https://healthcare.by/search.php?typ=707&sort=1&page=17
https://healthcare.by/search.php?

In [7]:
df = pd.DataFrame(rows)

In [8]:
df = df.drop_duplicates(subset="link").reset_index(drop=True)

## 5. Scrape addresses and websites

Each facility page is opened to collect its address and external website, when available.

In [9]:
rows = df.to_dict("records")

In [10]:
for row in rows:
    # Open the facility page
    response = requests.get(row["link"])
    doc = BeautifulSoup(response.text, "html.parser")

    # Find the first paragraph containing a <b> tag
    info = next(
        (p for p in doc.find_all("p") if p.find("b")),
        None
    )

    # Default values if the data is not available
    address = None
    website = None

    if info:
        # Extract all text parts from the paragraph
        text_parts = list(info.stripped_strings)

        # The first text after the facility name is the address
        if len(text_parts) > 1:
            address = text_parts[1].replace("\xa0", " ").strip()

        # Find an external website link
        website_tag = info.select_one(
            'a[href^="http://"], a[href^="https://"]'
        )

        if website_tag:
            website = website_tag.get("href")

    # Add the extracted values to the existing row
    row["address"] = address
    row["website"] = website

In [28]:
df = pd.DataFrame(rows)

## 6. Clean facility types

Additional abbreviations are used to classify records that were not recognized during the first pass. Records without a usable facility type or address are removed.

In [29]:
# Select rows where both facility type columns are missing
missing_type = (
    df["facility_type"].isna()
    & df["facility_type_eng"].isna()
)

# Find hospital abbreviations:
# ЦКРБ, ЦРБ, ЦГБ, or УБ
hospital_mask = (
    missing_type
    & df["facility_title"].str.contains(
        r"(?<!\w)(?:ЦРКБ|ЦРБ|ЦГБ|УБ)(?!\w)",
        case=False,
        na=False,
        regex=True
    )
)

# Update hospital values
df.loc[
    hospital_mask,
    ["facility_type", "facility_type_eng"]
] = ["Больница", "Hospital"]


# Recalculate the missing-value mask after updating hospitals
missing_type = (
    df["facility_type"].isna()
    & df["facility_type_eng"].isna()
)

# Find outpatient clinic names and abbreviations
ambulatory_mask = (
    missing_type
    & (
        df["facility_title"].str.contains(
            r"(?<!\w)(?:АВОП|ВАП|АОП)(?!\w)"
            r"|БСУ\s+с\s+ВА"
            r"|врача\s+общей\s+практики",
            case=False,
            na=False,
            regex=True
        )
        |
        # Match standalone "ВА" only when written in uppercase
        df["facility_title"].str.contains(
            r"(?<!\w)ВА(?!\w)",
            case=True,
            na=False,
            regex=True
        )
    )
)

# Update outpatient clinic values
df.loc[
    ambulatory_mask,
    ["facility_type", "facility_type_eng"]
] = ["Амбулатория", "Outpatient Clinic"]

In [30]:
# Remove rows where either facility type column is still missing
df = df.dropna(
    subset=["facility_type", "facility_type_eng"]
).reset_index(drop=True)

In [14]:
# # Remove rows where address column is still missing
# df = df.dropna(
#     subset=["address"]
# ).reset_index(drop=True)

## 7. Check and repair scraped addresses

Suspicious values are identified using address markers, postal codes, phone numbers, website patterns, and institution names. Only affected facility pages are scraped again.

In [31]:
import re

# Common address markers
address_markers = [
    "ул.",
    "пр.",
    "пр-т",
    "пер.",
    "пл.",
    "д.",
    "аг.",
    "агр.",
    "г.п.",
    "г.",
    "обл.",
    "р-н",
    "с/с",
    "п.",
]

# Build a regular expression for address markers
address_pattern = "|".join(
    re.escape(marker) for marker in address_markers
)

# Patterns that often indicate contact details or other non-address text
junk_pattern = (
    r"тел\.?|телефон|факс|"
    r"https?://|www\.|@|"
    r"\bГУЗ\b|\bУЗ\b"
)

# A Belarusian postal code usually contains 6 digits
postal_code_pattern = r"\b\d{6}\b"

# Address looks valid if it contains an address marker or postal code
looks_like_address = (
    df["address"].str.contains(
        address_pattern,
        case=False,
        na=False,
        regex=True
    )
    |
    df["address"].str.contains(
        postal_code_pattern,
        na=False,
        regex=True
    )
)

# Detect phone numbers, websites, email addresses, and institution names
looks_like_junk = df["address"].str.contains(
    junk_pattern,
    case=False,
    na=False,
    regex=True
)

# Find non-empty values that do not contain
# any address marker or postal code
suspicious_address_mask = (
    df["address"].notna()
    & ~looks_like_address
)

df.loc[
    suspicious_address_mask,
    ["facility_title", "link", "address", "website"]
]['address']

638                                 тел.: (02345) 3-66-88
1008                                                (ФАП)
1406    (Учреждение Здравоохранения «Поставская центра...
1858                                                (ЛПУ)
2104            (амбулатория, государственное учреждение)
2254                                тел.: (01713) 7-60-18
2626                               тел.: (0225)  43-68-52
2683                                        (амбулатория)
2800                      тел.: (0177) 95-61-04, 95-61-07
3096                                           (больница)
3358                                                 (УЗ)
3363            тел.: (0152) 44-21-80, 44-21-81, 68-10-97
3473                                            (vgcp.by)
Name: address, dtype: str

In [32]:
import re

# Address markers
address_markers = [
    "ул.", "пр.", "пр-т", "пер.", "пл.", "д.",
    "аг.", "агр.", "г.п.", "г.", "обл.",
    "р-н", "с/с", "п."
]

address_pattern = "|".join(
    re.escape(marker) for marker in address_markers
)

postal_code_pattern = r"\b\d{6}\b"

# Re-scrape only suspicious rows
for index in df.index[suspicious_address_mask]:
    response = requests.get(df.at[index, "link"], timeout=30)
    response.raise_for_status()

    doc = BeautifulSoup(response.text, "html.parser")

    # Find the main facility information paragraph
    info = next(
        (p for p in doc.find_all("p") if p.find("b")),
        None
    )

    if info is None:
        continue

    # Extract all text fragments from the paragraph
    text_parts = [
        text.replace("\xa0", " ").strip(" ()")
        for text in info.stripped_strings
    ]

    # Keep only fragments that look like an address
    address_candidates = [
        text for text in text_parts
        if (
            re.search(postal_code_pattern, text)
            or re.search(address_pattern, text, flags=re.IGNORECASE)
        )
    ]

    # Use the first address-like fragment
    if address_candidates:
        df.at[index, "address"] = address_candidates[0]

    # Extract the website if it is a clickable link
    website_tag = info.select_one(
        'a[href^="http://"], a[href^="https://"]'
    )

    if website_tag:
        df.at[index, "website"] = website_tag.get("href")

In [33]:
import re
import requests
from bs4 import BeautifulSoup

# Normalize text for comparison
title_text = df["facility_title"].fillna("").str.strip().str.lower()
address_text = df["address"].fillna("").str.strip().str.lower()

missing_address = (
    df["address"].isna()
    | address_text.eq("")
)

# Detect phone numbers or websites stored as an address
phone_or_website = df["address"].str.contains(
    r"^\s*(?:тел\.?|телефон|факс)"
    r"|https?://|www\."
    r"|(?:[a-z0-9-]+\.)+[a-z]{2,}",
    case=False,
    na=False,
    regex=True
)

# Detect cases where the address is identical to the facility title
same_as_title = (
    address_text.ne("")
    & address_text.eq(title_text)
)

# Detect institution descriptions without a postal code
institution_text = (
    ~df["address"].str.contains(r"\b\d{6}\b", na=False, regex=True)
    & df["address"].str.contains(
        r"\b(?:ГУЗ|УЗ|ЛПУ|ФАП|поликлиника|больница|амбулатория)\b"
        r"|учреждени[ея]\s+здравоохранения",
        case=False,
        na=False,
        regex=True
    )
)

# Select rows that need to be re-scraped
bad_address_mask = (
    missing_address
    | phone_or_website
    | same_as_title
    | institution_text
)

df.loc[
    bad_address_mask,
    ["facility_title", "address", "link"]
]

,facility_title,address,link
638,Золотушский ФАП,тел.: (02345) 3-66-88,https://healthcare.by/instinfo.php?orgnum=15855
2254,Дубровская врачебная амбулатория общей практики,тел.: (01713) 7-60-18,https://healthcare.by/instinfo.php?orgnum=14134
2626,Титовская амбулатория врача общей практики,тел.: (0225) 43-68-52,https://healthcare.by/instinfo.php?orgnum=24959
2800,Ганцевичская участковая больница,"тел.: (0177) 95-61-04, 95-61-07",https://healthcare.by/instinfo.php?orgnum=6175
2939,Краснослободская больница сестренского ухода,NaN,https://healthcare.by/instinfo.php?orgnum=6579
3358,Детская поликлиника г. Слоним,Детская поликлиника г. Слоним,https://healthcare.by/instinfo.php?orgnum=5905
3363,Детская стоматологическая поликлиника г. Гродно,Филиал учреждения здравоохранения «Центральная...,https://healthcare.by/instinfo.php?orgnum=5556


In [34]:
for index in df.index[bad_address_mask]:
    response = requests.get(df.at[index, "link"], timeout=30)
    response.raise_for_status()

    doc = BeautifulSoup(response.text, "html.parser")

    # Find the main facility information paragraph
    info = next(
        (p for p in doc.find_all("p") if p.find("b")),
        None
    )

    if info is None:
        df.at[index, "address"] = None
        continue

    # Extract separate text fragments from the paragraph
    text_parts = [
        text.replace("\xa0", " ").strip(" ()")
        for text in info.stripped_strings
    ]

    # Remove empty values, phone numbers and websites
    text_parts = [
        text for text in text_parts
        if text
        and not re.search(
            r"^(?:тел\.?|телефон|факс)"
            r"|https?://|www\."
            r"|(?:[a-z0-9-]+\.)+[a-z]{2,}$",
            text,
            flags=re.IGNORECASE
        )
    ]

    # First choice: a line containing a six-digit postal code
    address_candidates = [
        text for text in text_parts
        if re.search(r"\b\d{6}\b", text)
    ]

    # Fallback: a line containing an address marker and a number
    if not address_candidates:
        address_candidates = [
            text for text in text_parts
            if re.search(
                r"\b(?:ул\.|пр-т|пр\.|пер\.|пл\.|д\.|аг\.|агр\.|"
                r"г\.п\.|г\.|обл\.|р-н|с/с|п\.)",
                text,
                flags=re.IGNORECASE
            )
            and re.search(r"\d", text)
            and text.lower() != title_text.at[index]
        ]

    # Replace the incorrect value or use None if no address was found
    df.at[index, "address"] = (
        address_candidates[0]
        if address_candidates
        else None
    )

    # Extract the facility website when available
    website_tag = info.select_one(
        'a[href^="http://"], a[href^="https://"]'
    )

    if website_tag:
        df.at[index, "website"] = website_tag.get("href")

## 8. Save the scraped dataset

In [19]:
df.to_csv("output_data/healthcare_facilities_healthcare.by.csv")

# Geocoding

Google was used for the first geocoding pass. Yandex was then used to check and replace coordinates, especially for facilities in small settlements.

Store `GOOGLE_GEO_API_KEY`, `GOOGLE_PLACES_KEY`, and `YANDEX_API_KEY` in a local `.env` file.

In [35]:
import os
from dotenv import load_dotenv
from dotenv import dotenv_values
load_dotenv()

GOOGLE_GEO_API_KEY = os.environ["GOOGLE_GEO_API_KEY"]
GOOGLE_PLACES_KEY = os.environ["GOOGLE_PLACES_KEY"]

## 9. Initial geocoding with Google

In [21]:
from geopy.geocoders import GoogleV3

geolocator = GoogleV3(GOOGLE_GEO_API_KEY)

In [36]:
df["latitude"] = None
df["longitude"] = None

for i, address in df["address"].items():
    if pd.isna(address):
        address = df.loc[i, "facility_title"]

    location = geolocator.geocode(address,
        components={"country": "BY"},
        language="ru")

    if location:
        df.loc[i, "latitude"] = location.latitude
        df.loc[i, "longitude"] = location.longitude

## 10. Re-geocode facilities with Yandex

Saved Yandex results are loaded and applied first. New API requests are optional and disabled by default.

The same function can be used for FAPs, outpatient clinics, hospitals, or polyclinics by changing `FACILITY_TYPES`.

In [37]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2
from dotenv import load_dotenv


load_dotenv(override=True)
api_key = os.getenv("YANDEX_API_KEY")


# Calculate the distance between Google and Yandex coordinates in kilometers
def calculate_distance(lat1, lon1, lat2, lon2):
    earth_radius = 6371

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    latitude_difference = lat2 - lat1
    longitude_difference = lon2 - lon1

    a = (
        sin(latitude_difference / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(longitude_difference / 2) ** 2
    )

    return 2 * earth_radius * atan2(sqrt(a), sqrt(1 - a))


# Geocode one batch of facilities and return the comparison results
def geocode_yandex_batch(df, indices, api_key, delay=0.1):
    comparison_rows = []

    for i in indices:
        address = df.at[i, "address"]

        # Use facility title and settlement if the address is missing
        if pd.isna(address) or not str(address).strip():
            address = (
                str(df.at[i, "facility_title"])
                + ", "
                + str(df.at[i, "settlement"])
            )

        google_latitude = df.at[i, "latitude"]
        google_longitude = df.at[i, "longitude"]

        response = requests.get(
            "https://geocode-maps.yandex.ru/v1/",
            params={
                "apikey": api_key,
                "geocode": address,
                "format": "json",
                "lang": "ru_RU",
                "bbox": "23.1,51.2~32.8,56.3",
                "rspn": 1,
                "results": 1,
            },
            timeout=30,
        )

        response.raise_for_status()
        data = response.json()

        results = data[
            "response"
        ][
            "GeoObjectCollection"
        ][
            "featureMember"
        ]

        if results:
            geo_object = results[0]["GeoObject"]

            # Yandex returns longitude first, then latitude
            yandex_longitude, yandex_latitude = map(
                float,
                geo_object["Point"]["pos"].split()
            )

            metadata = geo_object[
                "metaDataProperty"
            ][
                "GeocoderMetaData"
            ]

            # Calculate distance only if Google coordinates exist
            if pd.notna(google_latitude) and pd.notna(google_longitude):
                distance_km = calculate_distance(
                    float(google_latitude),
                    float(google_longitude),
                    yandex_latitude,
                    yandex_longitude,
                )
            else:
                distance_km = None

            # Replace coordinates in the same original DataFrame row
            df.at[i, "latitude"] = yandex_latitude
            df.at[i, "longitude"] = yandex_longitude

            yandex_address = metadata.get("text")
            yandex_precision = metadata.get("precision")

        else:
            yandex_latitude = None
            yandex_longitude = None
            distance_km = None
            yandex_address = None
            yandex_precision = None

        comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "facility_type": df.at[i, "facility_type"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": yandex_latitude,
            "yandex_longitude": yandex_longitude,
            "distance_km": distance_km,
            "yandex_address": yandex_address,
            "yandex_precision": yandex_precision,
        })

        time.sleep(delay)

    return pd.DataFrame(comparison_rows)

### Load and apply saved Yandex results

This allows the notebook to reuse coordinates collected during earlier API batches without sending the requests again.

In [38]:
YANDEX_RESULTS_PATH = Path("gathered_yandex_coordinates.csv")

if YANDEX_RESULTS_PATH.exists():
    comparison_df = pd.read_csv(
        YANDEX_RESULTS_PATH,
        index_col=0
    )
    print("Saved Yandex results loaded:", len(comparison_df))
else:
    comparison_df = pd.DataFrame()
    print("No saved Yandex results found.")

Saved Yandex results loaded: 2577


In [40]:
def normalize_text(series):
    return (
        series
        .astype("string")
        .fillna("")
        .str.lower()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"[^\w\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def build_geocoding_address(data):
    address = (
        data["address"]
        .astype("string")
        .fillna("")
        .str.strip()
    )

    fallback_address = (
        data["facility_title"]
        .astype("string")
        .fillna("")
        .str.strip()
        + ", "
        + data["settlement"]
        .astype("string")
        .fillna("")
        .str.strip()
    )

    return address.mask(address.eq(""), fallback_address)


if not comparison_df.empty:
    # Keep successful saved Yandex results
    updates = (
        comparison_df
        .dropna(
            subset=[
                "yandex_latitude",
                "yandex_longitude",
            ]
        )
        .copy()
    )

    updates["title_key"] = normalize_text(
        updates["facility_title"]
    )

    updates["address_key"] = normalize_text(
        updates["address"]
    )

    # Empty keys cannot be matched safely
    updates = updates[
        updates["title_key"].ne("")
        & updates["address_key"].ne("")
    ].copy()

    # Keep the latest result for each facility-address pair
    updates = updates.drop_duplicates(
        subset=[
            "title_key",
            "address_key",
        ],
        keep="last",
    )

    # Prepare the current facility table
    current_facilities = df.copy()
    current_facilities["df_index"] = current_facilities.index

    current_facilities["match_address"] = (
        build_geocoding_address(current_facilities)
    )

    current_facilities["title_key"] = normalize_text(
        current_facilities["facility_title"]
    )

    current_facilities["address_key"] = normalize_text(
        current_facilities["match_address"]
    )

    valid_current = current_facilities[
        current_facilities["title_key"].ne("")
        & current_facilities["address_key"].ne("")
    ].copy()

    # Many current duplicate rows may match one saved Yandex result
    matched = valid_current.merge(
        updates[
            [
                "title_key",
                "address_key",
                "yandex_latitude",
                "yandex_longitude",
            ]
        ],
        on=[
            "title_key",
            "address_key",
        ],
        how="inner",
        validate="many_to_one",
    )

    matched = (
        matched
        .drop_duplicates(
            subset="df_index",
            keep="last",
        )
        .set_index("df_index")
    )

    df.loc[
        matched.index,
        "latitude",
    ] = pd.to_numeric(
        matched["yandex_latitude"],
        errors="coerce",
    )

    df.loc[
        matched.index,
        "longitude",
    ] = pd.to_numeric(
        matched["yandex_longitude"],
        errors="coerce",
    )

    # Find saved results that did not match any current row
    matched_keys = matched[
        [
            "title_key",
            "address_key",
        ]
    ].drop_duplicates()

    unmatched_updates = updates.merge(
        matched_keys,
        on=[
            "title_key",
            "address_key",
        ],
        how="left",
        indicator=True,
    )

    unmatched_updates = unmatched_updates[
        unmatched_updates["_merge"].eq("left_only")
    ].drop(columns="_merge")

    duplicate_rows_updated = matched.duplicated(
        subset=[
            "title_key",
            "address_key",
        ],
        keep=False,
    ).sum()

    print("Successful unique Yandex results:", len(updates))
    print("Current facility rows updated:", len(matched))
    print("Duplicate facility rows updated:", duplicate_rows_updated)
    print("Unmatched Yandex results:", len(unmatched_updates))

Successful unique Yandex results: 4
Current facility rows updated: 4
Duplicate facility rows updated: 0
Unmatched Yandex results: 0


### Optionally process a new Yandex batch

Set `RUN_NEW_YANDEX_BATCH` to `True`, choose one or more facility types, and set the batch size. Facilities already present in the saved results are skipped.

In [41]:
RUN_NEW_YANDEX_BATCH = False

FACILITY_TYPES = ["ФАП"]
BATCH_SIZE = 500
RANDOM_STATE = 42

In [ ]:
if RUN_NEW_YANDEX_BATCH:
    if not api_key:
        raise ValueError("YANDEX_API_KEY was not found in .env")

    if comparison_df.empty:
        checked_indices = set()
    else:
        checked_indices = set(comparison_df["df_index"])

    candidates = df[
        df["facility_type"].isin(FACILITY_TYPES)
        & ~df.index.isin(checked_indices)
    ]

    batch_size = min(BATCH_SIZE, len(candidates))

    batch_indices = candidates.sample(
        n=batch_size,
        random_state=RANDOM_STATE
    ).index

    new_comparison_df = geocode_yandex_batch(
        df=df,
        indices=batch_indices,
        api_key=api_key,
    )

    comparison_df = pd.concat(
        [comparison_df, new_comparison_df],
        ignore_index=True
    )

    comparison_df.to_csv(YANDEX_RESULTS_PATH)
    df.to_csv(
        "healthcare_with_coordinates_with_yandex.csv",
        index=False
    )

    print("Processed:", len(new_comparison_df))
    print(
        "Found by Yandex:",
        new_comparison_df["yandex_latitude"].notna().sum()
    )
    print(
        "Remaining unchecked facilities:",
        len(candidates) - len(new_comparison_df)
    )
else:
    print("New Yandex requests are disabled.")

## 11. Save the current geocoded dataset

In [44]:
df.to_csv(
    "output_data/healthcare_with_coordinates_with_yandex.csv",
    index=False
)

## 13. Check coordinates

Points outside the approximate bounding box of Belarus are listed for manual review.

In [45]:
latitude = pd.to_numeric(df["latitude"], errors="coerce")
longitude = pd.to_numeric(df["longitude"], errors="coerce")

outside_belarus = df[
    latitude.notna()
    & longitude.notna()
    & (
        (latitude < 51.2)
        | (latitude > 56.3)
        | (longitude < 23.1)
        | (longitude > 32.8)
    )
]

print("Points outside Belarus:", len(outside_belarus))

outside_belarus[
    [
        "facility_title",
        "facility_type",
        "address",
        "latitude",
        "longitude"
    ]
]

Points outside Belarus: 0


,facility_title,facility_type,address,latitude,longitude


In [ ]:
#df.to_csv("output_data/healthcare_with_coordinates_with_yandex.csv")

# Administrative boundaries

Facility points are spatially joined to the administrative boundary layer to add region and district names.

This notebook uses `gadm41_BLR.gpkg`, layer `ADM_ADM_2`.

In [52]:
import pandas as pd
import geopandas as gpd


# Keep all original columns
df_geo = df.copy()


# Ensure coordinates are numeric
df_geo["latitude"] = pd.to_numeric(
    df_geo["latitude"],
    errors="coerce",
)

df_geo["longitude"] = pd.to_numeric(
    df_geo["longitude"],
    errors="coerce",
)


# Create point geometry
# X = longitude, Y = latitude
df_gdf = gpd.GeoDataFrame(
    df_geo,
    geometry=gpd.points_from_xy(
        df_geo["longitude"],
        df_geo["latitude"],
    ),
    crs="EPSG:4326",
)


# Read administrative districts
adm2 = gpd.read_file(
    "map_data/administrative_boundaries_belarus_gadm41.gpkg",
    layer="ADM_ADM_2",
)

adm2 = adm2[
    ["NAME_1", "NAME_2", "geometry"]
].copy()


# Match coordinate systems
adm2 = adm2.to_crs(df_gdf.crs)


# Add region and district to every facility
df_gdf = gpd.sjoin(
    df_gdf,
    adm2,
    how="left",
    predicate="within",
)


# Remove technical join column
df_gdf = df_gdf.drop(
    columns="index_right",
    errors="ignore",
)


# Rename administrative columns
df_gdf = df_gdf.rename(
    columns={
        "NAME_1": "region",
        "NAME_2": "district",
    }
)


# Update the original DataFrame
df["region"] = df_gdf["region"]
df["district"] = df_gdf["district"]

ValueError: Columns must be same length as key

In [49]:
# Save GeoPackage with all columns and geometry
df_gdf.to_file(
    "output_data/healthcare_regions.gpkg",
    layer="healthcare",
    driver="GPKG",
    index=False,
)


# Save CSV with all original columns plus region and district
df_gdf.drop(columns="geometry").to_csv(
    "output_data/healthcare_regions.csv",
    index=False,
    encoding="utf-8-sig",
)

ValueError: GeoDataFrame cannot contain duplicated column names.